# Test doc Redis

In [1]:
# import redis 
# import pandas as pd

# # Tạo kết nối Redis
# r = redis.Redis(host='localhost', port=6379, db=15)

# # Hash key
# hash_key = 'OG_CRTO_MA10, MA20'

# # Đọc dữ liệu từ hash
# data = r.hgetall(hash_key)
# print('JSON:', data)

# if data:
# 	# Chuyển đổi dữ liệu từ bytes sang chuỗi nếu cần
# 	data = {key.decode('utf-8'): value.decode('utf-8') for key, value in data.items()}
# 	print('Dictionary:', data)
# 	# dataFrame = pd.DataFrame([data])
# 	# print('DataFrame:', dataFrame)

# Ham vao lenh

In [2]:
# Hàm để đặt một lệnh mua
from binance.client import Client
from binance.enums import *
import os

# Đặt API key và Secret key
api_key = '5d78mWKnHKtUDAttCD3BhTcNvquJxhFzbltdMbzxZu7ZmYUQ97ZdQM44zxCxQxxk' # K11_2
api_secret = 'ud0LM3iX1r7EEEBJCcdBTHgQQn9sIa0FTFRMt3MNN1jXHC9WZgRxkDGpjsUklEkZ' # K11_2

def orderSendCRTOFT(data):
    # Lấy dữ liệu từ từ điển
    datetime = data['Datetime']
    open_price = data['Open']
    close_price = data['Close']
    volume = data['Volume']    
    buy_signal = data['Buy_Signal']    
    sell_signal = data['Sell_Signal']
    symbol = data['Symbol']
    insertdate = data['Insertdate']
    
    # Khởi tạo client
    client = Client(api_key, api_secret)

    quantity = 150
    leverage = 49  # Đòn bẩy

    # Thay đổi đòn bẩy
    client.futures_change_leverage(symbol=symbol, leverage=leverage)
    # Thay đổi loại ký quỹ chéo (Cross) sang ký quỹ riêng lẻ (ISOLATED)
    # client.futures_change_margin_type(symbol=symbol, marginType='ISOLATED')

    # Lấy thông tin giá gần nhất của ETHUSDT
    current_price = client.get_symbol_ticker(symbol=symbol)
    entry_price = current_price["price"]

    # Chuyển đổi entry_price sang float
    entry_price = float(entry_price)
    
    allowed_precision = 3  # Giả sử độ chính xác tối đa là 6 chữ số thập phân

    order = None

    try:
        if buy_signal == 'True':
            print("Placing a BUY market order:", quantity)
            # Đặt Take Profit và Stop Loss (cần tính toán dựa trên chiến lược)
            take_profit = round(entry_price * 1.01, allowed_precision)
            stop_loss = round(entry_price * 0.98, allowed_precision)
            
            # Đặt lệnh mua
            buy_order = client.futures_create_order(
                symbol=symbol,
                side=SIDE_BUY,
                type=ORDER_TYPE_MARKET,
                quantity=quantity
            )           
            print("Buy order placed successfully:", buy_order)
            
            print("Placing a TAKE PROFIT market order.")
            tp_order = client.futures_create_order(
                symbol=symbol,
                side=SIDE_SELL,
                type=FUTURE_ORDER_TYPE_TAKE_PROFIT_MARKET ,
                timeInForce=TIME_IN_FORCE_GTC,
                quantity=quantity,
                stopPrice=take_profit,
                reduceOnly='true'
            )
            print("Take profit order placed successfully:", tp_order)

            print("Placing a STOP LOSS stop market order.")
            sl_order = client.futures_create_order(
                symbol=symbol,
                side=SIDE_SELL,
                type=FUTURE_ORDER_TYPE_STOP_MARKET,
                quantity=quantity,
                stopPrice=stop_loss,
                reduceOnly='true'
            )
            print("Stop loss order placed successfully:", sl_order)

        elif sell_signal == 'True':
            print("Placing a SELL market order.")
            # Đặt Take Profit và Stop Loss (cần tính toán dựa trên chiến lược)
            take_profit = round(entry_price * 0.99, allowed_precision)
            stop_loss = round(entry_price * 1.02, allowed_precision)
            
            # Đặt lệnh bán
            sell_order = client.futures_create_order(
                symbol=symbol,
                side=SIDE_SELL,
                type=ORDER_TYPE_MARKET,
                quantity=quantity
            )            
            print("Sell order placed successfully:", sell_order)
           
            print("Placing a TAKE PROFIT market order.")
            tp_order = client.futures_create_order(
                symbol=symbol,
                side=SIDE_BUY,
                type=FUTURE_ORDER_TYPE_TAKE_PROFIT_MARKET,
                timeInForce=TIME_IN_FORCE_GTC,
                quantity=quantity,
                stopPrice=take_profit,
                reduceOnly='true'
            )
            print("Take profit order placed successfully:", tp_order)

            print("Placing a STOP LOSS stop market order.")
            sl_order = client.futures_create_order(
                symbol=symbol,
                side=SIDE_BUY,
                type=FUTURE_ORDER_TYPE_STOP_MARKET,
                quantity=quantity,
                stopPrice=stop_loss,
                reduceOnly='true'
            )
            print("Stop loss order placed successfully:", sl_order)

    except Exception as e:
        print(f"An error occurred: {e}")

In [3]:
import redis

def entryCRTOFT(hash):
    # Xử lý dữ liệu ở đây
    print("Dữ liệu đã được xử lý")

    # Tạo kết nối Redis
    r = redis.Redis(host='localhost', port=6379, db=15)

    # Đọc dữ liệu từ hash
    data = r.hgetall(hash)
    
    if data:
        # Chuyển đổi dữ liệu từ bytes sang chuỗi nếu cần
        data = {key.decode('utf-8'): value.decode('utf-8') for key, value in data.items()}
        print('Redis:', data)
        # Xử lý dữ liệu data
        orderSendCRTOFT(data)
        
        # Xóa hash sau khi xử lý
        r.delete(hash)
        print(f"Hash '{hash}' đã được xóa khỏi Redis.")
    else:
        print(f"Không tìm thấy dữ liệu cho hash '{hash}'.")

# Schedule

In [ ]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_entryCRTOFT():
    entryCRTOFT('OG_CRTO_MA10, MA20_C98USDT')  # Gọi hàm entryCRTOFT với hash cụ thể

# Lên lịch hàm scan_market để chạy mỗi 15 phút
schedule.every(5).seconds.do(schedule_entryCRTOFT)

while True:
    schedule.run_pending()  # Hàm này được gọi liên tục để kiểm tra xem có công việc nào đã đến thời gian cần thực hiện hay không
    # time.sleep(1)

Dữ liệu đã được xử lý
Redis: {'Datetime': '2025-06-18T15:19:00', 'Symbol': 'C98USDT', 'Insertdate': '2025-06-18T22:19:24.999995', 'Open': '0.0458', 'High': '0.0459', 'Low': '0.0458', 'Close': '0.0459', 'Volume': '8941.6', 'MA10': '0.04591999999999999', 'MA20': '0.046060000000000066', 'Buy_Signal': 'False', 'Sell_Signal': 'True'}
Placing a SELL market order.
An error occurred: APIError(code=-2019): Margin is insufficient.
Hash 'OG_CRTO_MA10, MA20_C98USDT' đã được xóa khỏi Redis.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'OG_CRTO_MA10, MA20_C98USDT'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'OG_CRTO_MA10, MA20_C98USDT'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'OG_CRTO_MA10, MA20_C98USDT'.
Dữ liệu đã được xử lý
Redis: {'Datetime': '2025-06-18T15:20:00', 'Symbol': 'C98USDT', 'Insertdate': '2025-06-18T22:20:13.699838', 'Open': '0.046', 'High': '0.046', 'Low': '0.046', 'Close': '0.046', 'Volume': '138.5', 'MA10': '0.04590999999999999', 'MA20': '0.046055